# Solución — Proyecto Data Stream Processor

In [2]:
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.ch06.array_stack import ArrayStack
from goodrich.exceptions import Empty

## Clase 'Data Procesor'

In [ ]:
class DataProcessor:
    """Procesa un flujo de registros (sensor, variable, value)."""

    def __init__(self):
        self._queue = ArrayQueue()     
        self._history = ArrayStack()    
        self._redo_stack = ArrayStack() 
        self._state = []

    def _busca_indice(self, sensor, variable):
        """Ubica el índice de (sensor, variable) en el estado."""

        for i, (s, v, va) in enumerate(self._state):
          if s == sensor and v == variable:
              return i
        return None

    # Se creo este método porque otros 4 méotodos lo utilizan para hacer respectivos cambios y actualizaciones,
    # era mejor tenerlo como método por si la lógica que usabamos fallaba, hacer el cambio en un solo lugar y no
    # en todos, también le da limpieza a la clase en general. En tal caso de no existir el registro, retorna None,
    # para q más adelante otros métodos sepan como manejar el caso.


    def add(self, registro):
        """Agrega 'registro' a la cola de pendientes. No lo procesa."""

        if not isinstance(registro, tuple) or len(registro) != 3:
            raise ValueError("El registro no es válido")

        sensor, variable, valor = registro

        if not isinstance(valor, (int, float)):
            raise ValueError("El valor del registro debe ser numérico")

        self._queue.enqueue(registro)

    # Antes de añadir el registro, verificabamos que fuera un registro válido, y en caso de que no
    # lanza un ValueError que era el que más se asemejaba a la situación. Luego lo encola en un queue
    # que es donde estan todos los registros a ser procesados.


    def process_next(self):
        """Procesa el siguiente registro pendiente y actualiza el estado."""

        registro = self._queue.dequeue()
        sensor, variable, valor = registro
        idx = self._busca_indice(sensor, variable)

        if idx == None:
            self._state.append((sensor, variable, valor))
            cambio = ("creación", sensor, variable, valor)

        else:
            va_anterior = self._state[idx][2]
            self._state[idx] = (sensor, variable, valor)
            cambio = ("actualización", sensor, variable, va_anterior, valor)

        self._history.push(cambio)
        self._redo_stack = ArrayStack()
        return registro

    # Acá si procesa un registro a la vez, primero desencolandolo de la queue de pendientes, luego desempaqueta
    # las variables y se busca si ya existe el registro, en tal caso de no existir se agrega al historial como creación,
    # si existe lo que hace es actualizar su valor. También resetea el stack que se utiliza para el método .redo(), esto
    # solo por convención de como se maneja en otros sistemas de este estilo.


    def undo(self):
        """Deshace el último cambio aplicado por process_next()."""

        cambio = self._history.pop()
        self._revert(cambio)
        self._redo_stack.push(cambio)

    def _revert(self, cambio):
        accion = cambio[0]

        if accion == "creación":
            acc, sensor, variable, valor = cambio
            idx = self._busca_indice(sensor, variable)
            del self._state[idx]

        else:
            acc, sensor, variable, va_anterior, va_nuevo = cambio
            idx = self._busca_indice(sensor, variable)
            self._state[idx] = (sensor, variable, va_anterior)

    # Se creó un método auxiliar privado para darle limpieza al código, lo que hace primero es eliminar el cambio
    # del historial y sacar exactamente cual fue el cambio, luego se mira si fue "creación" o "actualización", dependiendo de
    # eso elimina directamente el registro en _state o devuelve al valor antiguo, por último este cambio deshecho con .undo()
    # queda guardado en un stack creado para .redo().


    def redo(self):
        """Vuelve a aplicar el último cambio deshecho con undo()."""

        cambio = self._redo_stack.pop()
        self._reapply(cambio)
        self._history.push(cambio)

    def _reapply(self, cambio):
        accion = cambio[0]
        if accion == "creación":
            acc, sensor, variable, value = cambio
            self._state.append((sensor, variable, value))
        else: 
            acc, sensor, variable, va_anterior, va_nueva = cambio
            idx = self._busca_indice(sensor, variable)
            self._state[idx] = (sensor, variable, va_nueva)

    # Se creó un método auxiliar privado para darle limpieza al código, lo que hace primero es extrar el cambio
    # del "historial de .undo()" y sacar exactamente cual fue el cambio, luego se mira si fue "creación" o "actualización", dependiendo de
    # eso vuelve a añadir directamente el registro en _state o devuelve al valor nuevo, por último este cambio hecho con .redo()
    # queda guardado en el historial nuevamente.


    def pending(self):
        """Número de registros que aún esperan ser procesados."""
        return len(self._queue)


    def current_value(self, sensor, variable):
        """Valor actual de (sensor, variable)."""
        idx = self._busca_indice(sensor, variable)
        if idx == None:
            raise KeyError("No registra.")
        return self._state[idx][2]


## Pruebas

In [4]:
## .add() y .pending()

dp = DataProcessor()

print("pending() antes de agregar:", dp.pending())
dp.add(("S01", "temperature", 23.5))
print("pending() después de agregar 1 registro:", dp.pending())

dp.add(("S02", "temperature", 25.1))
dp.add(("S01", "humidity", 61.2))
print("pending() después de agregar 3 registros en total:", dp.pending())

pending() antes de agregar: 0
pending() después de agregar 1 registro: 1
pending() después de agregar 3 registros en total: 3


In [ ]:
## .process_next() y .current_value()

dp = DataProcessor()
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

dp.add(A)
dp.add(B)
dp.add(C)

print("Procesado 1:", dp.process_next())  
print("valor actual:", dp.current_value("S01", "temperature"))

print("Procesado 2:", dp.process_next())   
print("valor actual:", dp.current_value("S01", "temperature"))

print("Procesado 3:", dp.process_next())
print("valor actual humidity:", dp.current_value("S01", "humidity"))

print("pending() al final:", dp.pending())

Procesado 1: ('S01', 'temperature', 20)
valor actual: 20
Procesado 2: ('S01', 'temperature', 25)
valor actual: 25
Procesado 3: ('S01', 'humidity', 60)
valor actual humidity: 60
pending() al final: 0


In [ ]:
## .undo()

dp = DataProcessor()
dp.add(("S01", "temperature", 20))
dp.process_next()
dp.add(("S01", "temperature", 25))
dp.process_next()
dp.add(("S01", "temperature", 30))
dp.process_next()

print("valor tras 3 procesos:", dp.current_value("S01", "temperature"))

dp.undo()
print("valor tras 1 undo:", dp.current_value("S01", "temperature"))     

dp.undo()
print("valor tras 2 undo:", dp.current_value("S01", "temperature"))     

dp.undo()
try:
    dp.current_value("S01", "temperature")
except KeyError:
    print("Tras el 3er undo, (S01, temperature) ya no existe: correcto")

In [ ]:
## .redo()

dp = DataProcessor()
dp.add(("S01", "temperature", 20))
dp.process_next()
dp.add(("S01", "temperature", 25))
dp.process_next()
dp.add(("S01", "temperature", 30))
dp.process_next()
print("valor inicial:", dp.current_value("S01", "temperature"))  

dp.undo()
print("tras undo():", dp.current_value("S01", "temperature"))   

dp.redo()
print("tras redo():", dp.current_value("S01", "temperature"))    

# Si se procesa un registro nuevo, el redo() pendiente se invalida
dp.undo()
dp.add(("S01", "humidity", 55))
dp.process_next()
try:
    dp.redo()
    assert False, "no debería quedar nada que rehacer"
except Empty:
    print("Un cambio nuevo invalida el redo() pendiente, como se esperaba")

## Pruebas Obligatorias

In [ ]:
# 1. pending() sobre un procesador vacío
dp = DataProcessor()
assert dp.pending() == 0
print("1. OK - pending() vacío == 0")

# 2. Agregar un registro
dp.add(("S01", "temperature", 23.5))
assert dp.pending() == 1
print("2. OK - agregar un registro")

# 3. Agregar varios registros
dp.add(("S02", "temperature", 25.1))
dp.add(("S01", "humidity", 61.2))
assert dp.pending() == 3
print("3. OK - agregar varios registros")

# 4. Verificar procesamiento FIFO
dp2 = DataProcessor()
A, B, C = ("S01", "t", 1), ("S02", "t", 2), ("S03", "t", 3)
dp2.add(A); dp2.add(B); dp2.add(C)
assert dp2.process_next() == A
assert dp2.process_next() == B
assert dp2.process_next() == C
print("4. OK - orden FIFO respetado")

# 5. Procesar un registro
dp3 = DataProcessor()
dp3.add(("S01", "temperature", 23.5))
registro = dp3.process_next()
assert registro == ("S01", "temperature", 23.5)
assert dp3.current_value("S01", "temperature") == 23.5
print("5. OK - procesar un registro")

# 6. Procesar varios registros
dp3.add(("S02", "temperature", 25.1))
dp3.add(("S01", "humidity", 61.2))
dp3.process_next()
dp3.process_next()
assert dp3.current_value("S02", "temperature") == 25.1
assert dp3.current_value("S01", "humidity") == 61.2
print("6. OK - procesar varios registros")

# 7. Actualizar una variable existente
dp3.add(("S01", "temperature", 27.0))
dp3.process_next()
assert dp3.current_value("S01", "temperature") == 27.0
print("7. OK - actualizar variable existente")

# 8. Consultar el valor actual
assert dp3.current_value("S01", "humidity") == 61.2
print("8. OK - consultar valor actual")

# 9. Realizar un undo()
antes = dp3.current_value("S01", "temperature")
dp3.undo()
despues = dp3.current_value("S01", "temperature")
assert antes == 27.0 and despues == 23.5
print("9. OK - undo() simple")

# 10. Realizar varios undo() consecutivos
dp3.undo()  # deshace creación de humidity
dp3.undo()  # deshace creación de S02/temperature
try:
    dp3.current_value("S02", "temperature")
    assert False, "no debería existir"
except KeyError:
    pass
print("10. OK - varios undo() consecutivos")

# 11. Procesar cuando la Queue está vacía
dp_vacio = DataProcessor()
try:
    dp_vacio.process_next()
    assert False, "debía lanzar Empty"
except Empty:
    print("11. OK - process_next() en cola vacía lanza Empty")

# 12. undo() cuando el historial está vacío
try:
    dp_vacio.undo()
    assert False, "debía lanzar Empty"
except Empty:
    print("12. OK - undo() en historial vacío lanza Empty")

# 13. Deshacer la creación de un dato que antes no existía
dp4 = DataProcessor()
dp4.add(("S05", "pressure", 101.3))
dp4.process_next()
assert dp4.current_value("S05", "pressure") == 101.3
dp4.undo()
try:
    dp4.current_value("S05", "pressure")
    assert False, "no debería existir tras el undo"
except KeyError:
    print("13. OK - undo() deshace la creación de un dato nuevo")

# 14. Varios cambios sobre la misma variable
dp5 = DataProcessor()
for v in (10, 20, 30, 40):
    dp5.add(("S01", "temperature", v))
    dp5.process_next()
assert dp5.current_value("S01", "temperature") == 40
dp5.undo()
assert dp5.current_value("S01", "temperature") == 30
dp5.undo()
assert dp5.current_value("S01", "temperature") == 20
print("14. OK - varios cambios sobre la misma variable")

# 15. Agregar un registro con formato incorrecto
dp6 = DataProcessor()
try:
    dp6.add(("S01", "temperature"))  # le falta value
    assert False, "debía rechazar el registro"
except ValueError:
    print("15. OK - registro con formato incorrecto es rechazado")

# 16. Agregar un registro cuyo valor no sea numérico
try:
    dp6.add(("S01", "temperature", "no numérico"))
    assert False, "debía rechazar el registro"
except ValueError:
    print("16. OK - registro con value no numérico es rechazado")

# 17. Consultar un sensor/variable que nunca haya sido procesado
try:
    dp6.current_value("S99", "no_existe")
    assert False, "debía lanzar KeyError"
except KeyError:
    print("17. OK - current_value() de una combinación nunca procesada lanza KeyError")

1. OK - pending() vacío == 0
2. OK - agregar un registro
3. OK - agregar varios registros
4. OK - orden FIFO respetado
5. OK - procesar un registro
6. OK - procesar varios registros
7. OK - actualizar variable existente
8. OK - consultar valor actual
9. OK - undo() simple
10. OK - varios undo() consecutivos
11. OK - process_next() en cola vacía lanza Empty
12. OK - undo() en historial vacío lanza Empty
13. OK - undo() deshace la creación de un dato nuevo
14. OK - varios cambios sobre la misma variable
15. OK - registro con formato incorrecto es rechazado
16. OK - registro con value no numérico es rechazado
17. OK - current_value() de una combinación nunca procesada lanza KeyError

Todas las pruebas obligatorias pasaron correctamente.


## Complejidad

**Explicación.**

- **`.add()` — O(1)** Ocasionalmente duplica el arreglo (_resize(), que es O(n)), pero ese costo se reparte entre las n inserciones anteriores.

- **`.process_next()` — O(n)** El costo lo impone ._find_index(), que en el peor caso recorre todo _state buscando (sensor, variable).

- **`.undo()` / `.redo()` — O(n).** El costo lo vuelve a imponer ._find_index() dentro de ._revert()/._reapply() para localizar la posición a modificar o eliminar en _state; del self._state[idx] en el peor caso también es O(n) porque puede recorrer/desplazar el resto de la lista.

- **`.pending()` — O(1).** Es una simple lectura de len(self._queue), devuelve directamente el contador _size que la propia clase mantiene actualizado.

- **`.current_value()` — O(n).** Depende enteramente de _find_index, que hace una búsqueda lineal sobre _state.